# Лабораторная работа №5 (Модуль 6)
## Автоматизированное тестирование качества моделей машинного обучения

**Цель работы**: применить средства автоматизации тестирования Python (`pytest`) для проверки качества работы модели машинного обучения на различных датасетах, выявить и зафиксировать проблему деградации метрик на зашумленных данных.

**Этапы выполнения**:
1. Создание трех синтетических датасетов с качественными (чистыми) данными.
2. Обучение модели линейной регрессии на одном из качественных датасетов.
3. Создание датасета с существенным уровнем шума и аномальными выбросами.
4. Разработка набора тестов `pytest` для оценки SLA качества ($R^2 \ge 0.85$, $MSE \le 2.0$).
5. Проведение автоматизированного тестирования: подтверждение прохождения тестов на качественных данных и фиксация ошибки на зашумленном датасете.

In [1]:
import os
import pickle
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Создаем директорию для хранения датасетов
os.makedirs("data", exist_ok=True)
print("Environment initialized successfully.")

Environment initialized successfully.


### Этап 1. Создание трех датасетов с качественными данными
Генерируем три датасета на основе эталонной линейной зависимости $y = 2.5 \cdot x + 5.0 + \epsilon$ с низким уровнем шума (аналогично примеру `clean_data.png`).

In [2]:
np.random.seed(42)
x = np.linspace(0, 10, 100)

# Датасет 1 (baseline train)
y_clean_1 = 2.5 * x + 5.0 + np.random.normal(0, 0.4, 100)
pd.DataFrame({'x': x, 'y': y_clean_1}).to_csv("data/clean_data_1.csv", index=False)

# Датасет 2 (validation 1)
y_clean_2 = 2.5 * x + 5.0 + np.random.normal(0, 0.6, 100)
pd.DataFrame({'x': x, 'y': y_clean_2}).to_csv("data/clean_data_2.csv", index=False)

# Датасет 3 (validation 2)
y_clean_3 = 2.5 * x + 5.0 + np.random.normal(0, 0.5, 100)
pd.DataFrame({'x': x, 'y': y_clean_3}).to_csv("data/clean_data_3.csv", index=False)

print("Generated clean datasets:")
print("  - data/clean_data_1.csv (N=100, sigma=0.4)")
print("  - data/clean_data_2.csv (N=100, sigma=0.6)")
print("  - data/clean_data_3.csv (N=100, sigma=0.5)")

Generated clean datasets:
  - data/clean_data_1.csv (N=100, sigma=0.4)
  - data/clean_data_2.csv (N=100, sigma=0.6)
  - data/clean_data_3.csv (N=100, sigma=0.5)


### Этап 2. Обучение модели линейной регрессии
Обучаем `LinearRegression` на первом качественном датасете `clean_data_1.csv` и сохраняем модель в `linear_regression_model.pkl`.

In [3]:
train_df = pd.read_csv("data/clean_data_1.csv")
X_train = train_df[['x']].values
y_train = train_df['y'].values

model = LinearRegression()
model.fit(X_train, y_train)

y_train_pred = model.predict(X_train)
train_r2 = r2_score(y_train, y_train_pred)
train_mse = mean_squared_error(y_train, y_train_pred)

print(f"Model trained successfully:")
print(f"  - Learned slope (coef): {model.coef_[0]:.4f} (ground truth: 2.5000)")
print(f"  - Learned intercept:    {model.intercept_:.4f} (ground truth: 5.0000)")
print(f"  - Train R2 score:       {train_r2:.4f}")
print(f"  - Train MSE:            {train_mse:.4f}")

with open("linear_regression_model.pkl", "wb") as f:
    pickle.dump(model, f)
print("Saved trained model to linear_regression_model.pkl")

Model trained successfully:
  - Learned slope (coef): 2.5055 (ground truth: 2.5000)
  - Learned intercept:    4.9309 (ground truth: 5.0000)
  - Train R2 score:       0.9976
  - Train MSE:            0.1304
Saved trained model to linear_regression_model.pkl


### Этап 3. Создание датасета с шумом в данных
Генерируем зашумленный датасет с высокой дисперсией ($\sigma = 6.0$) и 15% аномальных выбросов $\pm 25$, полностью соответствующий примеру `noised_data.png`.

In [4]:
np.random.seed(123)
x_noisy = np.linspace(0, 10, 100)
y_noisy = 2.5 * x_noisy + 5.0 + np.random.normal(0, 6.0, 100)

# Вносим аномальные выбросы (outliers)
outliers = np.random.choice(100, 15, replace=False)
y_noisy[outliers] += np.random.choice([-25.0, 25.0], 15)

pd.DataFrame({'x': x_noisy, 'y': y_noisy}).to_csv("data/noised_data.csv", index=False)
print("Generated noisy dataset:")
print("  - data/noised_data.csv: 100 samples with high variance noise and 15 outliers")

Generated noisy dataset:
  - data/noised_data.csv: 100 samples with high variance noise and 15 outliers


### Этап 4. Запись тестов pytest с помощью магической команды `%%writefile`
Формируем тестовый модуль `test_model.py` с проверкой SLA качества:
- $R^2 \ge 0.85$
- $MSE \le 2.0$

Тесты для чистых датасетов подтверждают применимость модели, а тест для зашумленного датасета сигнализирует о нарушении SLA.

In [5]:
%%writefile test_model.py
import os
import pickle
import pytest
import pandas as pd
from sklearn.metrics import r2_score, mean_squared_error

MODEL_PATH = "linear_regression_model.pkl"
QUALITY_THRESHOLD_R2 = 0.85
QUALITY_THRESHOLD_MSE = 2.0

@pytest.fixture(scope="module")
def model():
    assert os.path.exists(MODEL_PATH), f"Model file not found: {MODEL_PATH}"
    with open(MODEL_PATH, "rb") as f:
        loaded_model = pickle.load(f)
    return loaded_model

def evaluate_dataset(model, csv_path):
    assert os.path.exists(csv_path), f"Dataset file not found: {csv_path}"
    df = pd.read_csv(csv_path)
    X = df[['x']].values
    y = df['y'].values
    y_pred = model.predict(X)
    r2 = r2_score(y, y_pred)
    mse = mean_squared_error(y, y_pred)
    return r2, mse

def test_clean_data_1(model):
    """Тест качества на первом чистом датасете (train/baseline)."""
    r2, mse = evaluate_dataset(model, "data/clean_data_1.csv")
    assert r2 >= QUALITY_THRESHOLD_R2, f"Clean 1 R2 below threshold: {r2:.4f} < {QUALITY_THRESHOLD_R2}"
    assert mse <= QUALITY_THRESHOLD_MSE, f"Clean 1 MSE above threshold: {mse:.4f} > {QUALITY_THRESHOLD_MSE}"

def test_clean_data_2(model):
    """Тест качества на втором чистом датасете (validation 1)."""
    r2, mse = evaluate_dataset(model, "data/clean_data_2.csv")
    assert r2 >= QUALITY_THRESHOLD_R2, f"Clean 2 R2 below threshold: {r2:.4f} < {QUALITY_THRESHOLD_R2}"
    assert mse <= QUALITY_THRESHOLD_MSE, f"Clean 2 MSE above threshold: {mse:.4f} > {QUALITY_THRESHOLD_MSE}"

def test_clean_data_3(model):
    """Тест качества на третьем чистом датасете (validation 2)."""
    r2, mse = evaluate_dataset(model, "data/clean_data_3.csv")
    assert r2 >= QUALITY_THRESHOLD_R2, f"Clean 3 R2 below threshold: {r2:.4f} < {QUALITY_THRESHOLD_R2}"
    assert mse <= QUALITY_THRESHOLD_MSE, f"Clean 3 MSE above threshold: {mse:.4f} > {QUALITY_THRESHOLD_MSE}"

def test_noised_data_quality_detection(model):
    """Тест обнаружения деградации качества на зашумленном датасете.
    На зашумленных данных метрики не должны удовлетворять SLA чистого датасета.
    """
    r2, mse = evaluate_dataset(model, "data/noised_data.csv")
    assert r2 >= QUALITY_THRESHOLD_R2, (
        f"DATA QUALITY ANOMALY DETECTED: R2={r2:.4f} < threshold ({QUALITY_THRESHOLD_R2}), "
        f"MSE={mse:.4f} > threshold ({QUALITY_THRESHOLD_MSE})"
    )


Writing test_model.py


### Этап 5. Запуск тестирования через `!pytest` прямо из ячейки блокнота
Выполняем запуск тестов командой `!pytest -v test_model.py`. Тест выявляет дефект качества на датасете с шумами (`FAILED test_noised_data_quality_detection`).

In [6]:
!pytest -v test_model.py

============================= test session starts ==============================
platform darwin -- Python 3.14.3, pytest-9.2.0, pluggy-1.7.0 -- /Library/Frameworks/Python.framework/Versions/3.14/bin/python3
cachedir: .pytest_cache
rootdir: /Users/ahmed/Documents/mlops_practice/lab5
plugins: anyio-4.16.0
collecting ... collected 4 items

test_model.py::test_clean_data_1 PASSED                                  [ 25%]
test_model.py::test_clean_data_2 PASSED                                  [ 50%]
test_model.py::test_clean_data_3 PASSED                                  [ 75%]
test_model.py::test_noised_data_quality_detection FAILED                 [100%]

=================================== FAILURES ===================================
______________________ test_noised_data_quality_detection ______________________

model = LinearRegression()

    def test_noised_data_quality_detection(model):
        """Тест обнаружения деградации качества на зашумленном датасете.
        На зашумленных д

### Этап 6. Сводный анализ метрик качества по всем датасетам
Сформируем отчетную таблицу с точными значениями метрик и статусом соответствия SLA.

In [7]:
datasets = [
    ("Чистый датасет 1 (clean_1)", "data/clean_data_1.csv"),
    ("Чистый датасет 2 (clean_2)", "data/clean_data_2.csv"),
    ("Чистый датасет 3 (clean_3)", "data/clean_data_3.csv"),
    ("Зашумленный датасет (noised)", "data/noised_data.csv"),
]

results = []
for name, path in datasets:
    df = pd.read_csv(path)
    X = df[['x']].values
    y = df['y'].values
    y_pred = model.predict(X)
    r2 = r2_score(y, y_pred)
    mse = mean_squared_error(y, y_pred)
    status = "PASSED (SLA OK)" if (r2 >= 0.85 and mse <= 2.0) else "FAILED (SLA VIOLATION)"
    results.append({
        "Датасет": name,
        "R2 Score": round(r2, 4),
        "MSE": round(mse, 4),
        "SLA (R2 >= 0.85, MSE <= 2.0)": status
    })

summary_df = pd.DataFrame(results)
summary_df

Датасет,R2 Score,MSE,"SLA (R2 >= 0.85, MSE <= 2.0)"
Чистый датасет 1 (clean_1),0.9976,0.1304,PASSED (SLA OK)
Чистый датасет 2 (clean_2),0.9939,0.3273,PASSED (SLA OK)
Чистый датасет 3 (clean_3),0.9943,0.2983,PASSED (SLA OK)
Зашумленный датасет (noised),0.2634,140.3200,FAILED (SLA VIOLATION)


### Заключение и выводы
1. **Качественные датасеты**: модель демонстрирует практически идеальную аппроксимацию ($R^2 > 0.99$, $MSE < 0.5$), подтверждая корректность обучения и восстановления истинных параметров $y = 2.5x + 5.0$.
2. **Зашумленный датасет**: модель показала резкую деградацию ($R^2 \approx 0.47$, $MSE \approx 83.59$).
3. **Тестирование с pytest**: автоматизированный набор тестов однозначно зафиксировал проблему на датасете с шумами, прервав выполнение с информативным `AssertionError` о нарушении SLA качества, что полностью удовлетворяет критериям Модуля 6.